# Session 2 — Comparative pretraining (the long one)

Trains every model family at every pretraining-set size, all into ONE run
directory so the scaling curves are plotted in place. This is the longest
session by a wide margin; keep the tab alive.

## 1. Setup

In [ ]:
!git clone -b improve_transformer https://github.com/rifatozkurt/FullWaveformInversion
%cd FullWaveformInversion

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'{p.name}, {p.total_memory/1e9:.1f} GB')
    # A single adjoint evaluation allocates ~3 GB. Anything under ~8 GB
    # means running one job at a time and nothing else on the GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/fwi_thesis'
!mkdir -p {OUT}

## 2. Data

`extended/` (ids 0-14999) is training data; `eval/` (ids 15000-15999) is
held out. They do NOT overlap. Restore both from Drive if you have them
zipped there, otherwise generate (slow).

In [ ]:
# Restore from Drive (fast path)
!unzip -q -o {OUT}/data/extended.zip -d /content/  || echo 'no extended.zip'
!unzip -q -o {OUT}/data/eval.zip     -d /content/  || echo 'no eval.zip'
!ls /content/extended | head -3 ; ls /content/eval | head -3

# --- alternative: generate instead (hours) ---
# !python scripts/generate_train_data_colab.py \
#     --config configs/config_final.yaml --output-dir /content/extended \
#     --start-case-id 0 --number-of-cases 15000 --case-batch-size 4 --no-overwrite

## 3a. U-Net and SegFormer at every sample count

Sample counts come from `comparative_pretraining.sample_counts` in the
config (250/500/1000/5000/10000/15000). Both families share seeds and
sample ids at every size, so the comparison is paired.

In [ ]:
# All three arms across the full sweep. ImageNet initialisation is now
# trained at every size, not just 15000: the interesting question is whether
# out-of-domain pretraining substitutes for in-domain data, and that matters
# most where in-domain data is scarcest.
#
# --overwrite is required: models/final already contains the previous run's
# checkpoints, and the runner refuses to clobber them silently.
!python scripts/temp_pretraining.py \
    --config configs/config_final.yaml \
    --data-dir /content/extended \
    --output-dir models/final \
    --run-dir runs/final/comparative_pretraining \
    --models unet,segformer,segformer_imagenet \
    --overwrite

## 3b. SegFormer with ImageNet initialization

Only at 15000. This answers a different question from the scaling curve —
whether out-of-domain pretraining substitutes for in-domain data — so it
is compared against the random-init SegFormer at the same size.

### Checkpoint inventory

Session 3 needs all of these.

In [ ]:
import pathlib
for p in sorted(pathlib.Path('models/final').iterdir()):
    print(f'{p.stat().st_size/1e6:8.1f} MB  {p.name}')

## 4. Save everything to Drive

`runs/` holds every history, CSV and figure; `models/` holds the
checkpoints. Zip both so a disconnect does not lose the session.

In [ ]:
SESSION = 'session2'
!mkdir -p {OUT}/{SESSION}
!zip -qr /content/runs.zip runs
!cp /content/runs.zip {OUT}/{SESSION}/runs.zip
!zip -qr /content/models.zip models
!cp /content/models.zip {OUT}/{SESSION}/models.zip
print('saved to', OUT + '/' + SESSION)

## 5. Look at the figures before you disconnect

In [ ]:
from IPython.display import Image, display
import pathlib
for p in sorted(pathlib.Path('runs/final').rglob('report/*.png')):
    print(p)
    display(Image(str(p)))